# SpliceTransformer Splice Site Prediction

![SpliceTransformer Splice Site Prediction](https://proto-bio.github.io/proto-assets/images/tool/splice_transformer/hero.png)

This notebook demonstrates `run_splice_transformer`, which predicts tissue-specific splice site locations from raw DNA sequence. Given a target sequence flanked by genomic context, the model returns per-position probabilities for three splice categories (neither, acceptor, donor) along with 15 tissue-specific usage channels, making it useful for variant interpretation, synthetic intron design, and analysis of tissue-specific alternative splicing.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("splice_transformer")
display_overview("splice_transformer")
display_docs_section("splice_transformer", "Background")

# SpliceTransformer

SpliceTransformer is a deep learning model for predicting [splice sites](https://en.wikipedia.org/wiki/RNA_splicing) with tissue-specific resolution. It identifies splice donors (5' splice sites) and acceptors (3' splice sites) across 15 human tissues, enabling analysis of [alternative splicing](https://en.wikipedia.org/wiki/Alternative_splicing) patterns and tissue-specific isoform usage. The model uses [transformer](https://en.wikipedia.org/wiki/Transformer_(deep_learning_architecture)) architecture with long-range sequence context for accurate splice site prediction.

## Background

**What is splicing?**
[Pre-mRNA splicing](https://en.wikipedia.org/wiki/RNA_splicing) removes [introns](https://en.wikipedia.org/wiki/Intron) and joins [exons](https://en.wikipedia.org/wiki/Exon):
- **Donor site (5' splice site)**: End of exon, beginning of intron (GT dinucleotide)
- **Acceptor site (3' splice site)**: End of intron, beginning of exon (AG dinucleotide)
- **Branch point**: Internal intron sequence for lariat formation

**Tissue-specific splicing:**
Different tissues express different splice isoforms:
- Tissue-specific [splicing factors](https://en.wikipedia.org/wiki/Splicing_factor)
- Alternative exon inclusion/exclusion
- Alternative 5'/3' splice site usage

**The 15 tissues:**
| Index | Tissue |
|-------|--------|
| 0 | Adipose tissue |
| 1 | Blood |
| 2 | Blood vessel |
| 3 | Brain |
| 4 | Colon |
| 5 | Heart |
| 6 | Kidney |
| 7 | Liver |
| 8 | Lung |
| 9 | Muscle |
| 10 | Nerve |
| 11 | Small intestine |
| 12 | Skin |
| 13 | Spleen |
| 14 | Stomach |

## Available tools

In [2]:
display_available_tools("splice_transformer")

- **`run_splice_transformer()`** — Tissue-specific splicing prediction using SpliceTransformer

### `run_splice_transformer`

Given a target DNA sequence flanked by left and right genomic context, `run_splice_transformer` uses a transformer architecture trained on RNA-seq data from 15 human tissues to predict per-position splice site probabilities. The output tensor has 18 channels: three splice type channels (neither, acceptor, donor) that sum to 1.0 at every position, followed by 15 tissue-specific usage channels covering adipose, blood, brain, heart, liver, and other tissues. This structure makes it straightforward to identify canonical splice sites and to compare their predicted usage across tissues.

In [3]:
import numpy as np

from proto_tools import (
    SPLICE_TISSUE_CHANNEL_INDEX,
    SpliceTransformerType,
    SpliceTransformerConfig,
    SpliceTransformerInput,
    run_splice_transformer,
)

In [4]:
# Display input docs
display_api_reference("splice_transformer", "input", "run_splice_transformer")

from pathlib import Path
from Bio import SeqIO

# Load human beta-globin (HBB) genomic sequence (NG_059281.1)
# HBB is a small, well-characterized gene with 3 exons and 2 introns
fasta_path = Path("./hbb_genomic.fasta")
record = SeqIO.read(fasta_path, "fasta")
full_seq = str(record.seq)
print(f"Gene: HBB (Homo sapiens hemoglobin subunit beta)")
print(f"Accession: {record.id}")
print(f"Total length: {len(full_seq):,}bp")

# Targets must be exactly 1000 bp; left/right contexts exactly 4000 bp.
target_start = 4700
target_end = 5700
target_seq = full_seq[target_start:target_end]
left_ctx = full_seq[target_start - 4000:target_start]
right_ctx = full_seq[target_end:target_end + 4000]

print(f"\nTarget: positions {target_start}-{target_end} ({len(target_seq)}bp)")
print(f"Context: {len(left_ctx)}bp left, {len(right_ctx)}bp right")

inputs = SpliceTransformerInput(
    target_seqs=[target_seq],
    left_contexts=[left_ctx],
    right_contexts=[right_ctx],
)

**Input** — `SpliceTransformerInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `target_seqs` | `list[str]` | required | Sequence(s) on which to make splicing predictions |
| `left_contexts` | `list[str]` | required | Sequence(s) of the left context. Must be the same length as target_seqs |
| `right_contexts` | `list[str]` | required | Sequence(s) of the right context. Must be the same length as target_seqs |

Gene: HBB (Homo sapiens hemoglobin subunit beta)
Accession: NG_059281.1
Total length: 10,106bp

Target: positions 4700-5700 (1000bp)
Context: 4000bp left, 4000bp right


In [5]:
# Display config docs
display_api_reference("splice_transformer", "config", "run_splice_transformer")

config = SpliceTransformerConfig(
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=True,
)

**Config** — `SpliceTransformerConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `verbose` | `int` | `0` | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| `device` | `str` | `'cuda'` | Device to run the model on |
| `timeout` | `int` | `600` | Maximum execution time in seconds |
| `seed` | `int \| None` | `None` | Random seed for reproducible results. Some cacheable tools gate cache on this field. |

In [6]:
# Run the prediction
result = run_splice_transformer(inputs, config)

Running inference.py (one-shot) with device=cuda:0


Running run_splice_transformer [00:00]

In [7]:
# Display output docs
display_api_reference("splice_transformer", "output", "run_splice_transformer")

pred = np.array(result.prediction)
print(f"Prediction shape: {pred.shape}")
print(f"  batch={pred.shape[0]}, target_length={pred.shape[1]}, channels={pred.shape[2]}")

**Output** — `SpliceTransformerOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `prediction` | `list[list[list[float]]]` | required | Matrix of [batch, target_length, 18] |

Prediction shape: (1, 1000, 18)
  batch=1, target_length=1000, channels=18


## Interpreting Predictions

The prediction tensor has 18 channels. Channels 0 through 2 correspond to splice type probabilities: "neither" (not a splice site), "acceptor", and "donor". Channels 3 through 17 provide tissue-specific splice usage predictions for 15 human tissues. The splice type channels sum to 1.0 at each position, representing a probability distribution over the three categories.

In [8]:
pred = np.array(result.prediction[0])  # First (only) sequence

# Splice type probabilities
neither_probs = pred[:, SpliceTransformerType.NEITHER.value]
acceptor_probs = pred[:, SpliceTransformerType.ACCEPTOR.value]
donor_probs = pred[:, SpliceTransformerType.DONOR.value]

print(f"Max acceptor probability: {acceptor_probs.max():.4f} at position {acceptor_probs.argmax()}")
print(f"Max donor probability: {donor_probs.max():.4f} at position {donor_probs.argmax()}")
print(f"Mean 'neither' probability: {neither_probs.mean():.4f}")

Max acceptor probability: 0.9914 at position 572
Max donor probability: 0.9857 at position 441
Mean 'neither' probability: 0.9960


## Finding High-Confidence Splice Sites

A threshold on acceptor and donor probabilities identifies positions where the model is confident a splice site exists. A threshold of 0.5 typically captures the strongest splice sites while filtering noise.

In [9]:
threshold = 0.5

acceptor_sites = np.where(acceptor_probs > threshold)[0]
donor_sites = np.where(donor_probs > threshold)[0]

print(f"High-confidence acceptor sites (>{threshold}): {len(acceptor_sites)}")
if len(acceptor_sites) > 0:
    print(f"  Positions: {acceptor_sites[:10]}..." if len(acceptor_sites) > 10 else f"  Positions: {acceptor_sites}")

print(f"High-confidence donor sites (>{threshold}): {len(donor_sites)}")
if len(donor_sites) > 0:
    print(f"  Positions: {donor_sites[:10]}..." if len(donor_sites) > 10 else f"  Positions: {donor_sites}")

High-confidence acceptor sites (>0.5): 2
  Positions: [332 572]
High-confidence donor sites (>0.5): 2
  Positions: [441 794]


## Tissue-Specific Analysis

Channels 3 through 17 provide splice usage predictions for 15 human tissues including brain, heart, liver, and muscle. Comparing these channels reveals tissue-specific alternative splicing, where the same splice site is used at different rates in different tissues. This is biologically important because alternative splicing is a major mechanism for generating protein diversity from a single gene.

In [10]:
pred = np.array(result.prediction[0])
for tissue_name, channel_idx in SPLICE_TISSUE_CHANNEL_INDEX.items():
    if channel_idx is None:  # AVERAGE is handled separately via channel slice 3:
        continue
    tissue_pred = pred[:, channel_idx]
    print(f"{tissue_name:20s}: mean={tissue_pred.mean():.4f}, max={tissue_pred.max():.4f}")

ADIPOSE_TISSUE      : mean=0.0026, max=0.6767
BLOOD               : mean=0.0014, max=0.3695
BLOOD_VESSEL        : mean=0.0024, max=0.6270
BRAIN               : mean=0.0026, max=0.6755
COLON               : mean=0.0028, max=0.7136
HEART               : mean=0.0023, max=0.6058
KIDNEY              : mean=0.0029, max=0.7335
LIVER               : mean=0.0021, max=0.5648
LUNG                : mean=0.0030, max=0.7633
MUSCLE              : mean=0.0020, max=0.5354
NERVE               : mean=0.0028, max=0.7103
SMALL_INTESTINE     : mean=0.0030, max=0.7714
SKIN                : mean=0.0026, max=0.6632
SPLEEN              : mean=0.0025, max=0.6588
STOMACH             : mean=0.0027, max=0.7100


## Export Results

Predictions can be exported to NumPy format for downstream analysis, including integration with variant effect prediction pipelines or visualization in genome browsers.

In [11]:
result.export("splice_transformer_predictions", export_path="./example_output", file_format="npy")
print("Predictions exported to ./example_output/splice_transformer_predictions.npy")

Predictions exported to ./example_output/splice_transformer_predictions.npy
